| [0.0 Data Collection](00_data_collection.ipynb) | [1.0 Preprocessing](01_data_preprocessing.ipynb) | 2.0 Spatial Autocorrelation | [3.0 MGWR](03_mgwr_analysis.ipynb) | [4.0 ML Classification](04_ml_classification.ipynb) |

# 2.0 | Spatial Autocorrelation Analysis

This notebook assesses whether road traffic accident rates exhibit spatial clustering across England.  
The analysis proceeds through:

1. **Queen contiguity** spatial weights construction
2. **Global Moran's I** — is there overall spatial clustering?
3. **Local Moran's I (LISA)** — where are the hot/cold spots?
4. **Yearly sensitivity analysis** — are patterns stable across 2021-2024?

In [ ]:
import sys
sys.path.insert(0, '..')

import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from src.utils.config import (
    MSOA_ANALYSIS_GPKG, PROCESSED_DIR, YEARS,
    FIGURES_DIR, FIGURE_DPI
)

plt.rcParams.update({
    'font.family': 'serif',
    'figure.dpi': 150,
    'axes.grid': True,
    'grid.alpha': 0.3
})

## 2.1 | Load and Inspect Data

In [ ]:
gdf = gpd.read_file(MSOA_ANALYSIS_GPKG)
gdf = gdf[gdf['accident_rate_per_10k'] > 0].copy()

print(f"Loaded {len(gdf):,} MSOAs (after removing zero-rate areas)")
print(f"CRS: {gdf.crs}")
print(f"\nAccident rate summary:")
gdf['accident_rate_per_10k'].describe()

### 2.1 | Figure 1 | Accident rate choropleth

A visual inspection of the spatial distribution of accident rates reveals apparent clustering,  
with elevated rates in urban centres and along major transport corridors.

In [ ]:
from src.visualization.maps import plot_choropleth
plot_choropleth(gdf)

# Inline display
fig, ax = plt.subplots(figsize=(10, 12))
gdf.plot(column='accident_rate_per_10k', cmap='YlOrRd', scheme='quantiles', k=5,
         legend=True, ax=ax, edgecolor='face', linewidth=0.1,
         legend_kwds={'loc': 'lower left', 'title': 'Rate per 10k (Quantiles)'})
ax.set_title('Accident Rate per 10,000 Population (2021-2024)', fontweight='bold', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

## 2.2 | Spatial Weights Matrix

A **Queen contiguity** weights matrix defines spatial neighbours as MSOAs sharing a boundary  
or vertex. This captures both edge and corner adjacencies.

In [ ]:
from src.analysis.spatial_autocorr import build_spatial_weights

w = build_spatial_weights(gdf)

print("Spatial Weights Summary")
print("─" * 40)
print(f"  Observations:     {w.n:,}")
print(f"  Mean neighbours:  {w.mean_neighbors:.1f}")
print(f"  Min neighbours:   {w.min_neighbors}")
print(f"  Max neighbours:   {w.max_neighbors}")
print(f"  Islands:          {len(w.islands)}")
if w.islands:
    print(f"  Island indices:   {w.islands[:10]}{'...' if len(w.islands) > 10 else ''}")

## 2.3 | Global Moran's I

Global Moran's I tests whether accident rates across all MSOAs exhibit statistically significant  
spatial clustering. Values range from -1 (dispersion) through 0 (random) to +1 (clustering).

**Hypotheses**:  
- H₀: Accident rates are randomly distributed across MSOAs  
- H₁: Accident rates exhibit significant spatial autocorrelation

In [ ]:
from src.analysis.spatial_autocorr import global_morans_i

global_result = global_morans_i(gdf, variable='accident_rate_per_10k', w=w)

print("Global Moran's I Test Results")
print("═" * 40)
print(f"  Moran's I:   {global_result['I']:.4f}")
print(f"  Expected I:  {global_result['expected_I']:.4f}")
print(f"  Z-score:     {global_result['z_score']:.4f}")
print(f"  p-value:     {global_result['p_value']:.6f}")
print("═" * 40)

if global_result['p_value'] < 0.001:
    print("\n→ Result: Highly significant spatial autocorrelation detected (p < 0.001).")
    print("  Accident rates are NOT randomly distributed — neighbouring MSOAs")
    print("  tend to have similar accident rates.")
elif global_result['p_value'] < 0.05:
    print("\n→ Result: Significant spatial autocorrelation (p < 0.05).")
else:
    print("\n→ Result: No significant spatial autocorrelation detected.")

### 2.3 | Figure 2 | Moran scatter plot

The Moran scatter plot displays standardised accident rates (x-axis) against their spatial lag  
(weighted average of neighbours, y-axis). The slope of the fitted line equals Moran's I.  
Points in the upper-right (HH) and lower-left (LL) quadrants indicate positive spatial autocorrelation.

In [ ]:
from esda.moran import Moran

y = gdf['accident_rate_per_10k'].values
mi = Moran(y, w, permutations=999)

# Standardise
y_std = (y - y.mean()) / y.std()
wy_std = np.array([sum(w[i][j] * y_std[j] for j in w[i]) for i in range(len(y_std))])

# Save to file
from src.visualization.statistical_plots import plot_moran_scatter
plot_moran_scatter(y_std, wy_std, mi.I, mi.p_sim)

# Inline display
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(y_std, wy_std, s=4, alpha=0.3, color='steelblue')

# Regression line
m, b = np.polyfit(y_std, wy_std, 1)
x_line = np.linspace(y_std.min(), y_std.max(), 100)
ax.plot(x_line, m * x_line + b, color='red', linewidth=2, label=f"slope = {m:.4f}")

# Quadrant lines
ax.axhline(0, color='grey', ls='--', lw=0.8)
ax.axvline(0, color='grey', ls='--', lw=0.8)

# Quadrant labels
ax.text(0.95, 0.95, 'HH', transform=ax.transAxes, fontsize=14, fontweight='bold',
        ha='right', va='top', color='red', alpha=0.5)
ax.text(0.05, 0.05, 'LL', transform=ax.transAxes, fontsize=14, fontweight='bold',
        ha='left', va='bottom', color='blue', alpha=0.5)
ax.text(0.95, 0.05, 'HL', transform=ax.transAxes, fontsize=14, fontweight='bold',
        ha='right', va='bottom', color='orange', alpha=0.5)
ax.text(0.05, 0.95, 'LH', transform=ax.transAxes, fontsize=14, fontweight='bold',
        ha='left', va='top', color='lightblue', alpha=0.5)

ax.set_xlabel('Standardised Accident Rate (z)', fontsize=12)
ax.set_ylabel('Spatial Lag of Accident Rate (Wz)', fontsize=12)
ax.set_title(f"Moran's I = {mi.I:.4f}  (p = {mi.p_sim:.4f}, 999 permutations)",
             fontweight='bold', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 2.4 | Local Moran's I (LISA)

While Global Moran's I confirms overall clustering, Local Indicators of Spatial Association (LISA)  
identify **where** clusters are located:

| Cluster | Meaning |
|---------|--------|
| **HH** (High-High) | Hot spot: high rate MSOA surrounded by high rate neighbours |
| **LL** (Low-Low) | Cold spot: low rate MSOA surrounded by low rate neighbours |
| **HL** (High-Low) | Spatial outlier: high rate surrounded by low rate |
| **LH** (Low-High) | Spatial outlier: low rate surrounded by high rate |
| **NS** | Not significant at α = 0.05 |

In [ ]:
from src.analysis.spatial_autocorr import local_morans_i

lisa_gdf = local_morans_i(gdf, variable='accident_rate_per_10k', w=w)

print("LISA Cluster Distribution")
print("─" * 35)
cluster_counts = lisa_gdf['lisa_cluster'].value_counts()
for cluster, count in cluster_counts.items():
    pct = count / len(lisa_gdf) * 100
    print(f"  {cluster:5s}  {count:5,d}  ({pct:.1f}%)")

### 2.4 | Figure 3 | LISA cluster map

The LISA cluster map identifies statistically significant spatial clusters of road traffic accidents.  
HH clusters (red) indicate accident hot spots; LL clusters (blue) indicate cold spots.

In [ ]:
from src.visualization.maps import plot_lisa_clusters
plot_lisa_clusters(lisa_gdf)

# Inline display
COLORS = {'HH': '#d7191c', 'LL': '#2c7bb6', 'HL': '#fdae61', 'LH': '#abd9e9', 'NS': '#d3d3d3'}
LABELS = {
    'HH': 'High-High (Hot Spot)',
    'LL': 'Low-Low (Cold Spot)',
    'HL': 'High-Low (Outlier)',
    'LH': 'Low-High (Outlier)',
    'NS': 'Not Significant'
}

fig, ax = plt.subplots(figsize=(10, 12))
patches = []
for cluster, colour in COLORS.items():
    subset = lisa_gdf[lisa_gdf['lisa_cluster'] == cluster]
    if not subset.empty:
        subset.plot(ax=ax, color=colour, edgecolor='face', linewidth=0.1)
        patches.append(mpatches.Patch(color=colour, label=LABELS[cluster]))

ax.legend(handles=patches, loc='lower left', fontsize=10, framealpha=0.9)
ax.set_title('LISA Clusters — Accident Rate per 10,000 (2021-2024)',
             fontweight='bold', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

## 2.5 | Yearly Sensitivity Analysis

To assess the robustness of spatial patterns, Global Moran's I is computed for each year separately.  
**Note**: 2021 is the first post-COVID recovery year and may exhibit atypical patterns due to  
reduced traffic volumes and changed commuting behaviour.

In [ ]:
from src.analysis.spatial_autocorr import sensitivity_analysis

yearly_df = sensitivity_analysis()

print("Yearly Global Moran's I")
print("═" * 55)
print(f"{'Year':>6s}  {'Moran I':>10s}  {'Z-score':>10s}  {'p-value':>10s}  {'Sig.':>6s}")
print("─" * 55)
for _, row in yearly_df.iterrows():
    sig = '***' if row['p_value'] < 0.001 else '**' if row['p_value'] < 0.01 else '*' if row['p_value'] < 0.05 else 'ns'
    print(f"{int(row['year']):>6d}  {row['morans_I']:>10.4f}  {row['z_score']:>10.4f}  {row['p_value']:>10.6f}  {sig:>6s}")
print("═" * 55)
print("\n→ Spatial autocorrelation is significant across all years,")
print("  confirming the robustness of spatial clustering patterns.")

yearly_df

### 2.5 | Figure 4 | Yearly Moran's I comparison

In [ ]:
from src.visualization.statistical_plots import plot_yearly_moran_comparison
plot_yearly_moran_comparison(yearly_df)

# Inline bar chart
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c' if y == 2021 else '#3498db' for y in yearly_df['year']]
bars = ax.bar(yearly_df['year'].astype(int).astype(str), yearly_df['morans_I'],
              color=colors, edgecolor='white', width=0.6)

for bar, val in zip(bars, yearly_df['morans_I']):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel("Moran's I", fontsize=12)
ax.set_title("Global Moran's I by Year (2021 = post-COVID recovery)",
             fontweight='bold', fontsize=13)
ax.set_ylim(0, max(yearly_df['morans_I']) * 1.2)
plt.tight_layout()
plt.show()

### 2.5 | Figure 5 | LISA clusters by year

Comparing LISA cluster maps across years to assess whether hot spots persist or shift over time.

In [ ]:
yearly_lisa = {}
for year in YEARS:
    path = PROCESSED_DIR / f"msoa_yearly_{year}.gpkg"
    if path.exists():
        y_gdf = gpd.read_file(path)
        y_gdf = y_gdf[y_gdf['accident_rate_per_10k'] > 0].copy()
        yearly_lisa[year] = local_morans_i(y_gdf, variable='accident_rate_per_10k')
        print(f"{year}: {len(y_gdf):,} MSOAs → "
              f"HH={sum(yearly_lisa[year]['lisa_cluster']=='HH')}, "
              f"LL={sum(yearly_lisa[year]['lisa_cluster']=='LL')}")

In [ ]:
if yearly_lisa:
    from src.visualization.maps import plot_lisa_yearly_comparison
    plot_lisa_yearly_comparison(yearly_lisa)

    # Inline panel plot
    fig, axes = plt.subplots(1, len(yearly_lisa), figsize=(6 * len(yearly_lisa), 8))
    if len(yearly_lisa) == 1:
        axes = [axes]

    for idx, (year, lgdf) in enumerate(sorted(yearly_lisa.items())):
        ax = axes[idx]
        for cluster, colour in COLORS.items():
            subset = lgdf[lgdf['lisa_cluster'] == cluster]
            if not subset.empty:
                subset.plot(ax=ax, color=colour, edgecolor='face', linewidth=0.1)
        note = ' (post-COVID)' if year == 2021 else ''
        ax.set_title(f'{year}{note}', fontweight='bold', fontsize=12)
        ax.axis('off')

    plt.suptitle('LISA Clusters by Year', fontweight='bold', fontsize=14)
    plt.tight_layout()
    plt.show()

## 2.6 | Summary

**Key findings from spatial autocorrelation analysis:**

1. **Global Moran's I** confirms statistically significant positive spatial autocorrelation  
   in accident rates — neighbouring MSOAs tend to have similar accident rates.

2. **LISA analysis** reveals distinct HH (hot spot) clusters concentrated in urban areas  
   and LL (cold spot) clusters in rural regions.

3. **Temporal robustness** — spatial patterns are consistent across all four years (2021-2024),  
   though 2021 shows slightly different patterns attributable to post-COVID recovery.

The presence of significant spatial autocorrelation in OLS residuals (tested in the next notebook)  
justifies the use of spatially-explicit models such as GWR and MGWR.

Proceed to **Notebook 03** for MGWR analysis.

---
| [0.0 Data Collection](00_data_collection.ipynb) | [1.0 Preprocessing](01_data_preprocessing.ipynb) | 2.0 Spatial Autocorrelation | [3.0 MGWR](03_mgwr_analysis.ipynb) | [4.0 ML Classification](04_ml_classification.ipynb) |